# ex03 · nb03 — 인터랙티브 실시간 놀이터 🎮

이 노트북은 **"고쳐서 바로 던져보는"** 개발 루프 그 자체가 주제야. 코드 예제라기보다,
**MoveItPy 를 켜둔 커널을 놀이터 삼아 팔을 실시간으로 쿡쿡 찔러보는** 워크플로우 시연이지.

## 왜 이게 그렇게 강력하냐면

기존 루프는 이랬어:

> **파일 고침 → `Ctrl-C` → `ros2 launch ...` → 30초 대기 → 로그 구경 → 아 오타났네 → 다시 처음부터**

이 노트북에선 그 루프가 이렇게 **붕괴**해:

> **셀 고침 → `Shift+Enter` → 즉시 팔이 움직임** ✨

핵심은 **커널이 살아있다**는 거야. 아래 설정 셀을 한 번 실행하면 `MoveItPy` 객체가
커널 메모리에 상주하고, 그 뒤로:

- `move_group` 도, 컨트롤러도 **절대 안 죽어**. 세션 내내 그대로 붙어 있음.
- 셀을 고쳐서 `Shift+Enter` 만 하면 **그 순간** 목표가 계획·실행돼.
- **`colcon` 재빌드 없음. `ros2 launch` 재시작 없음.** 파이썬 코드는 커널 안에서 즉석 재정의되니까.

> ⚠️ **재시작이 필요한 유일한 경우**: URDF/SRDF/컨트롤러(ros2_control) 같은 **로봇 정의 자체**를
> 바꿨을 때뿐이야. 그건 런치가 노드에 주입하는 파라미터라 커널만으론 못 바꿔. 그 외 순수 파이썬 로직은
> 전부 셀에서 즉석으로 갈아끼울 수 있어.

## 시작 전 체크 ✅

- 이 노트북은 **런치 스택이 먼저 떠 있어야** 돌아가. 별도 터미널에서
  `ros2 launch ... ex03_moveit_py_notebook.launch.py` 를 띄우고 **약 30초** 기다린 뒤 (move_group/컨트롤러 준비) 아래 설정 셀을 실행해.
- 🔒 **설정 셀(다음 두 셀)은 커널당 딱 한 번만** 실행할 것. `MoveItPy` 는 커널당 하나만 살 수 있어.
  두 번 실행하면 노드 이름 충돌로 꼬여. 다시 처음부터 하고 싶으면 **커널을 재시작**하고 한 번만 실행해.
- 🔒 한 번에 **노트북/커널 하나만** 띄워. 동시에 두 개 커널이 `MoveItPy` 를 잡으면 서로 싸워.


In [1]:
import time
import numpy as np
from geometry_msgs.msg import PoseStamped
from moveit.core.kinematic_constraints import construct_joint_constraint
from moveit.core.robot_state import RobotState
from moveit.planning import MoveItPy, MultiPipelinePlanRequestParameters
from rclpy.logging import get_logger
import rclpy
from moveit_py_params import build_moveit_config_dict

if not rclpy.ok():
    rclpy.init()

logger = get_logger("nb03_playground")
robot = MoveItPy(node_name="moveit_py", config_dict=build_moveit_config_dict())
arm = robot.get_planning_component("arm")
gripper = robot.get_planning_component("gripper")
robot_model = robot.get_robot_model()
print("놀이터 준비 완료 — 아래 헬퍼 셀 실행 후, 마지막 섹션에서 값만 바꿔가며 Shift+Enter")

놀이터 준비 완료 — 아래 헬퍼 셀 실행 후, 마지막 섹션에서 값만 바꿔가며 Shift+Enter


[INFO] [1784809459.555609062] [moveit_786748536.moveit.py.cpp_initializer]: Initialize rclcpp
[INFO] [1784809459.555649515] [moveit_786748536.moveit.py.cpp_initializer]: Initialize node parameters
[INFO] [1784809459.555658876] [moveit_786748536.moveit.py.cpp_initializer]: Initialize node and executor
[INFO] [1784809459.560341295] [moveit_786748536.moveit.py.cpp_initializer]: Spin separate thread
[INFO] [1784809459.565112699] [moveit_786748536.moveit.ros.rdf_loader]: Loaded robot model in 0.00465489 seconds
[INFO] [1784809459.565149255] [moveit_786748536.moveit.core.robot_model]: Loading robot model 'agx_arm'...
[INFO] [1784809459.565160221] [moveit_786748536.moveit.core.robot_model]: No root/virtual joint specified in SRDF. Assuming fixed joint
[INFO] [1784809459.679894053] [moveit_786748536.moveit.kinematics.kdl_kinematics_plugin]: Joint weights for group 'arm': 1 1 1 1 1 1
[INFO] [1784809460.117014619] [moveit_786748536.moveit.ros.planning_scene_monitor]: Publishing maintained planni

## 🧰 헬퍼 함수 — 이 노트북의 심장

여기가 진짜 핵심이야. 짧고 재사용 가능한 함수 몇 개만 커널에 등록해두면, 그 다음부터는
`go_named("home")`, `go_joints(joint1=0.5)`, `grip("gripper_open")` 같은 **한 줄**로 팔을 조종해.

패턴은 전부 동일해: **목표 세팅 → `plan()` → 되면 `execute()`**. 이 지겨운 3단계를 `_run()`
하나로 감싸버려서, 위쪽 함수들은 "무엇을" 만 신경 쓰고 "어떻게" 는 신경 안 써도 되게 했어.

> 💡 이 셀도 그냥 파이썬 함수 정의라, 나중에 함수 동작이 맘에 안 들면 **여기서 고치고 `Shift+Enter`**
> → 커널 안 정의가 즉석 교체돼. 아래 놀이터 셀들은 자동으로 새 정의를 쓰게 돼. 이게 REPL 개발의 맛이지.

In [2]:
def _run(component, sleep_time=0.5):
    """목표가 세팅된 planning_component 를 계획→실행. 성공 여부(bool) 반환."""
    result = component.plan()
    if result:
        robot.execute(result.trajectory, controllers=[])  # controllers=[] → 자동 선택
    else:
        logger.error("계획 실패 (도달 불가/충돌?)")
    time.sleep(sleep_time)
    return bool(result)


def go_named(name, group=None):
    "SRDF 이름 상태로 이동. group 기본은 arm. (arm: home / gripper: gripper_open 등)"
    comp = group or arm
    comp.set_start_state_to_current_state()
    comp.set_goal_state(configuration_name=name)
    return _run(comp)


def go_joints(**joints):
    "관절 목표로 이동. 예: go_joints(joint1=0.5, joint2=0.3). 안 준 관절은 0.0 으로 채움."
    arm.set_start_state_to_current_state()
    gs = RobotState(robot_model)
    # RobotState 는 모든 관절이 기본 0.0 으로 정의돼 있음. 안전하게 arm 6관절을
    # 명시적으로 0.0 으로 깔고, 사용자가 준 값만 덮어써서 부분 dict 도 예측 가능하게.
    positions = {f"joint{i}": 0.0 for i in range(1, 7)}
    positions.update({k: float(v) for k, v in joints.items()})
    gs.joint_positions = positions
    jc = construct_joint_constraint(
        robot_state=gs,
        joint_model_group=robot_model.get_joint_model_group("arm"),
    )
    arm.set_goal_state(motion_plan_constraints=[jc])
    return _run(arm)


def grip(name):
    "그리퍼 이름 상태: gripper_open / gripper_half / gripper_close"
    gripper.set_start_state_to_current_state()
    gripper.set_goal_state(configuration_name=name)
    return _run(gripper)


print("헬퍼 등록 완료: go_named / go_joints / grip  (그리고 내부용 _run)")

헬퍼 등록 완료: go_named / go_joints / grip  (그리고 내부용 _run)


## 🔎 현재 상태 들여다보기 (introspection)

인터랙티브 개발의 절반은 **"지금 로봇이 어디 있냐"** 를 즉석에서 확인하는 거야. 계획을 보내기 전에
현재 관절값을 찍어보고, 목표를 보낸 뒤엔 tcp_link 가 실제로 어디로 갔는지 FK(순기구학)로 확인해.

아래 셀은 `planning_scene_monitor` 의 **읽기 전용 스냅샷**을 떠서:

- 6개 관절의 현재 각도(rad)
- `tcp_link` 의 현재 월드 위치(x, y, z, base_link 기준)

를 출력해. 팔을 움직인 **직후에 이 셀을 다시 `Shift+Enter`** 하면 결과가 어떻게 바뀌는지 바로 보여.
이게 놀이터에서 "찔러보고 → 확인하고 → 또 찔러보는" 리듬이야.

In [3]:
# 현재 상태 스냅샷 — 움직인 뒤 다시 실행하면 값이 바뀐다.
try:
    psm = robot.get_planning_scene_monitor()
    with psm.read_only() as scene:
        st = scene.current_state
        st.update()  # FK 등 파생값 갱신
        cur = {j: round(st.joint_positions[j], 3)
               for j in ["joint1", "joint2", "joint3", "joint4", "joint5", "joint6"]}
        print("현재 관절:", cur)
        tf = st.get_global_link_transform("tcp_link")  # 4x4 동차변환
        print("tcp_link 위치:", np.round(np.asarray(tf)[:3, 3], 4))
except Exception as exc:
    print(f"상태 조회 실패({type(exc).__name__}: {exc})")
    print("→ 런치 스택/컨트롤러가 다 떴는지, 설정 셀을 실행했는지 확인.")

현재 관절: {'joint1': 0.0, 'joint2': 0.0, 'joint3': 0.0, 'joint4': 0.0, 'joint5': 0.0, 'joint6': 0.0}
tcp_link 위치: [0.1981 0.     0.2256]


## 🕹️ 라이브 포킹 — **여기부터가 핵심**

여기부터가 이 노트북의 진짜 놀이터야. **아래 셀들은 전부 한 줄짜리**고, 값만 바꿔서
`Shift+Enter` 하면 **즉시 팔이 움직여**. `colcon` 재빌드도, 런치 재시작도 없음.

숫자를 조금씩 바꿔가며 감을 잡아봐. 관절 크기는 대략 **±1.0 rad 이내**로 유지하면 리미트 안 넘고 안전해.
움직인 뒤엔 위의 🔎 introspection 셀을 다시 실행해서 어디로 갔는지 확인하는 습관을 들여.

**1) 홈 자세로** — 언제든 여기로 돌아오면 기준점.

In [4]:
go_named("home")

[WARN] [1784809460.908268596] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809460.908407680] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809460.908470616] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809460.908474738] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809460.908484480] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809460.908497661] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809460.908573563] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809460.908925251] [moveit_7

True

**2) 오른쪽으로 살짝** — joint1 을 +로.

In [5]:
go_joints(joint1=0.6, joint2=0.4, joint3=-0.5)

[WARN] [1784809461.607955412] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809461.608067553] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809461.608090856] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809461.608093848] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809461.608100779] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809461.608108117] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809461.608146186] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809461.608280940] [moveit_7

True

**3) 왼쪽으로 살짝** — joint1 부호만 뒤집음. 숫자 하나 바꿔서 대칭 확인.

In [6]:
go_joints(joint1=-0.6, joint2=0.4, joint3=-0.5)

[WARN] [1784809463.691992433] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809463.692128488] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809463.692153441] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809463.692156561] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809463.692185920] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809463.692193963] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809463.692231352] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809463.692370254] [moveit_7

True

**4) 그리퍼 열기** 👐

In [7]:
grip("gripper_open")

[WARN] [1784809466.419049266] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809466.419207863] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809466.419247195] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809466.419255587] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809466.419272239] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809466.419292611] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809466.419341956] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'gripper' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809466.419509021] [move

True

**5) 그리퍼 닫기** ✊ — 중간은 `grip("gripper_half")`.

In [8]:
grip("gripper_close")

[WARN] [1784809468.411692447] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809468.411830883] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809468.411858821] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809468.411861985] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809468.411889699] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809468.411898191] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809468.411925370] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'gripper' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809468.412048019] [move

True

**6) 팔꿈치+손목만 살짝** — 안 준 관절은 헬퍼가 0.0 으로 채우니 이 두 개만 신경 쓰면 됨.

In [9]:
go_joints(joint2=0.5, joint3=-0.7, joint5=0.3)

[WARN] [1784809470.397825215] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809470.397988826] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809470.398024096] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809470.398032328] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809470.398049596] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809470.398068490] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809470.398125914] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809470.398315633] [moveit_7

True

**7) 자유 실험 칸** — 여기다 아무 값이나 넣고 굴려봐. (예시는 아주 작은 움직임)

In [10]:
go_joints(joint1=0.2, joint4=0.3, joint6=0.4)

[WARN] [1784809472.483132098] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809472.483317795] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809472.483357982] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809472.483366110] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809472.483413881] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809472.483436735] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809472.483503721] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809472.483695502] [moveit_7

True

## Cartesian(포즈) 공간 라이브 포킹 — 그리퍼 EEF 기준

관절(joint) 대신 **tcp_link 의 3D 위치(x,y,z) + 방향(쿼터니언)** 으로 목표를 주면 MoveIt 이 IK 로 관절값을 풀어준다. 실제 작업은 대부분 이 Cartesian 사고로 한다.

> **기준점 = 그리퍼 파지 끝단(EEF).** moveit_py_params.py 의 tcp_offset_xyz 를 `"0.0 0.0 0.1425"` 로 잡아 tcp_link 를 link6 에서 +z 0.1425m(=플랜지→손끝) 앞으로 뺐다. 따라서 아래 go_pose/current_pose 의 좌표·RViz 의 tcp_link 는 모두 **그리퍼 끝단** 기준이다. (이 값은 런치가 읽으므로 바꾸면 **1회 리런치** 필요. 손끝보다 살짝 안쪽 파지점을 쓰려면 0.10~0.12 로 줄여도 됨.)

아래 헬퍼 셀을 한 번 실행해 `go_pose / nudge / current_pose` 를 등록한 뒤, **기준 포즈에서 숫자를 조금씩 더하고 빼며** `Shift+Enter` 하면 된다.

> IK 목표가 **도달 불가/특이점**이면 계획이 실패한다(`계획 실패` 출력). 값을 기준 포즈 쪽으로 되돌리면 된다.

In [11]:
# Cartesian(포즈) 헬퍼 — 관절 대신 tcp_link 의 (x,y,z)+방향으로 조종
from scipy.spatial.transform import Rotation as R  # 회전행렬 -> 쿼터니언


def current_pose():
    """현재 tcp_link 의 (position[x,y,z], quaternion[x,y,z,w]) 를 base_link 기준으로 반환."""
    psm = robot.get_planning_scene_monitor()
    with psm.read_only() as scene:
        st = scene.current_state
        st.update()
        T = np.asarray(st.get_global_link_transform("tcp_link"))  # 4x4 동차변환
    pos = T[:3, 3]
    quat = R.from_matrix(T[:3, :3]).as_quat()  # [x, y, z, w]
    return pos, quat


def go_pose(x, y, z, qx=None, qy=None, qz=None, qw=None, sleep_time=0.5):
    """tcp_link 를 (x,y,z)+쿼터니언 목표로 IK 계획->실행.
    쿼터니언을 생략(None)하면 '현재 방향을 그대로 유지'하고 위치만 바꾼다."""
    if qx is None:  # 방향 생략 -> 현재 tcp 방향 그대로
        _, q = current_pose()
        qx, qy, qz, qw = (float(v) for v in q)
    pose = PoseStamped()
    pose.header.frame_id = "base_link"
    pose.pose.position.x = float(x)
    pose.pose.position.y = float(y)
    pose.pose.position.z = float(z)
    pose.pose.orientation.x = float(qx)
    pose.pose.orientation.y = float(qy)
    pose.pose.orientation.z = float(qz)
    pose.pose.orientation.w = float(qw)
    arm.set_start_state_to_current_state()
    arm.set_goal_state(pose_stamped_msg=pose, pose_link="tcp_link")
    print(f"목표 pos=({x:.4f}, {y:.4f}, {z:.4f})  "
          f"quat=({qx:.4f}, {qy:.4f}, {qz:.4f}, {qw:.4f})")
    return _run(arm, sleep_time=sleep_time)


def nudge(dx=0.0, dy=0.0, dz=0.0, sleep_time=0.5):
    """현재 tcp 위치에서 (dx,dy,dz)[m] 만큼만 직교 이동(방향 유지).
    값을 조금씩 더하고 빼며 감 잡기 좋은 함수."""
    pos, quat = current_pose()
    return go_pose(pos[0] + dx, pos[1] + dy, pos[2] + dz,
                   *(float(v) for v in quat), sleep_time=sleep_time)


print("Cartesian 헬퍼 등록: current_pose / go_pose / nudge")


Cartesian 헬퍼 등록: current_pose / go_pose / nudge


**1) 기준 포즈로** — 네가 준 그 값. 여기서부터 시작.

In [12]:
# 기준 포즈 — 이 값에서 조금씩 더하고 빼면서 놀면 됨
go_pose(0.1158, 0.0, 0.3133,
        0.0, 0.8633, 0.0, 0.5047)


목표 pos=(0.1158, 0.0000, 0.3133)  quat=(0.0000, 0.8633, 0.0000, 0.5047)


[WARN] [1784809474.780159829] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809474.780285765] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809474.780330809] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809474.780333774] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809474.780341591] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809474.780349507] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809474.780389598] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809474.780560416] [moveit_7

False

**2) 지금 포즈 찍어보기** — 움직인 뒤 다시 실행하면 값이 바뀜.

In [13]:
# 지금 tcp 가 어디 있나 — 움직인 뒤 다시 실행하면 값이 바뀜
pos, quat = current_pose()
print("pos :", np.round(pos, 4).tolist())
print("quat:", np.round(quat, 4).tolist())


pos : [0.1941, 0.0378, 0.2301]
quat: [0.1729, 0.6496, 0.3238, 0.6658]


**3) 값 조금씩 더하고 빼기** — 바로 네가 말한 그 방식. x/z 를 ±3cm.

In [14]:
# 위 기준값에서 조금씩 더하고 빼기 (방향 quat 은 그대로 놓음)
go_pose(0.1158 + 0.03,   # x: 앞으로 +3cm
        0.0   + 0.00,    # y
        0.3133 - 0.03,   # z: 아래로 -3cm
        0.0, 0.8633, 0.0, 0.5047)


목표 pos=(0.1458, 0.0000, 0.2833)  quat=(0.0000, 0.8633, 0.0000, 0.5047)


[WARN] [1784809476.366085295] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809476.366260623] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809476.366300556] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809476.366309305] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809476.366328211] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809476.366348523] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809476.366415048] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809476.366633645] [moveit_7

True

**4) 방향 유지 + 위치만 nudge** — 쿼터니언 신경 안 쓰고 축 방향으로만 살짝.

In [15]:
# 방향은 유지하고 위치만 상대 이동 (미터). 부호/축만 바꿔가며 찔러봐.
nudge(dz=0.05)     # 5cm 위로
# nudge(dx=0.04)   # 앞으로
# nudge(dy=0.04)   # 옆으로


목표 pos=(0.1456, 0.0001, 0.3334)  quat=(-0.0025, 0.8624, 0.0032, 0.5062)


[WARN] [1784809479.012318984] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809479.012430863] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809479.012463323] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809479.012466341] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809479.012473772] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809479.012481906] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809479.012514901] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809479.012677031] [moveit_7

True

**5) 자유 Cartesian 칸** — 아무 값이나 넣고 굴려봐.

In [30]:
# 자유 Cartesian 칸 — 여기서 마음껏. 값만 바꿔 Shift+Enter.
go_pose(0.3, 0.0, 0.15,
        0.0, 0.8633, 0.0, 0.5047)


목표 pos=(0.3000, 0.0000, 0.1500)  quat=(0.0000, 0.8633, 0.0000, 0.5047)


[WARN] [1784809592.658997778] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809592.659162039] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809592.659199948] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809592.659208349] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809592.659229096] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809592.659249280] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809592.659310710] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809592.659527899] [moveit_7

True

In [31]:
# 자유 Cartesian 칸 — 여기서 마음껏. 값만 바꿔 Shift+Enter.
go_pose(0.2, 0.0, 0.0,
        0.0, 0.8633, 0.0, 0.0)


목표 pos=(0.2000, 0.0000, 0.0000)  quat=(0.0000, 0.8633, 0.0000, 0.0000)


[WARN] [1784809596.926161700] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809596.926377642] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809596.926425355] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809596.926434192] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809596.926460338] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809596.926482563] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809596.926542988] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809596.926821819] [moveit_7

True

In [32]:
# 자유 Cartesian 칸 — 여기서 마음껏. 값만 바꿔 Shift+Enter.
go_pose(0.2, 0.0, 0.1,
        0.0, 0.8633, 0.0, 0.0)


목표 pos=(0.2000, 0.0000, 0.1000)  quat=(0.0000, 0.8633, 0.0000, 0.0000)


[WARN] [1784809612.669069356] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809612.669268303] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809612.669313075] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809612.669322003] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809612.669347063] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809612.669369996] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809612.669433502] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809612.669654314] [moveit_7

True

In [39]:
# 자유 Cartesian 칸 — 여기서 마음껏. 값만 바꿔 Shift+Enter.
go_pose(0.3, -0.1, 0.1,
        0.0, 0.8633, 0.0, 0.0)


목표 pos=(0.3000, -0.1000, 0.1000)  quat=(0.0000, 0.8633, 0.0000, 0.0000)


[WARN] [1784809666.279647107] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809666.279767075] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809666.279795120] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809666.279798036] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809666.279805141] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809666.279812503] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809666.279846032] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809666.280014984] [moveit_7

True

In [ ]:
# 자유 Cartesian 칸 — 여기서 마음껏. 값만 바꿔 Shift+Enter.
go_pose(0.2, 0.0, 0.05,
        0.0, 0.8633, 0.0, 0.0)
go_pose(0.2, 0.0, 0.0,
        0.0, 0.8633, 0.0, 0.0)
go_pose(0.2, 0.2, 0.0,
        0.0, 0.8633, 0.0, 0.0)
go_pose(0.3, 0.2, 0.0,
        0.0, 0.8633, 0.0, 0.0)
go_pose(0.3, 0.0, 0.0,
        0.0, 0.8633, 0.0, 0.0)
go_pose(0.2, 0.0, 0.0,
        0.0, 0.8633, 0.0, 0.0)

목표 pos=(0.2000, 0.0000, 0.0500)  quat=(0.0000, 0.8633, 0.0000, 0.0000)


[WARN] [1784809805.674340493] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809805.674510639] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809805.674548867] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809805.674557564] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809805.674577292] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809805.674597860] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809805.674655646] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809805.674862335] [moveit_7

목표 pos=(0.2000, 0.0000, 0.0000)  quat=(0.0000, 0.8633, 0.0000, 0.0000)


[WARN] [1784809807.819372012] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809807.819521765] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809807.819557200] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809807.819565317] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809807.819581977] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809807.819601354] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809807.819656912] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809807.819892429] [moveit_7

목표 pos=(0.2000, 0.2000, 0.0000)  quat=(0.0000, 0.8633, 0.0000, 0.0000)


[WARN] [1784809809.359308413] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809809.359522441] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809809.359561316] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809809.359569258] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809809.359586208] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809809.359605843] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809809.359661788] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809809.359852859] [moveit_7

목표 pos=(0.3000, 0.2000, 0.0000)  quat=(0.0000, 0.8633, 0.0000, 0.0000)


[WARN] [1784809811.699376093] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809811.699546240] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809811.699583976] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809811.699592011] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809811.699609433] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809811.699629204] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809811.699688599] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809811.699886637] [moveit_7

목표 pos=(0.3000, 0.0000, 0.0000)  quat=(0.0000, 0.8633, 0.0000, 0.0000)


[WARN] [1784809813.434387376] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809813.434543918] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809813.434579084] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809813.434587166] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809813.434642193] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809813.434663953] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809813.434719834] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809813.434911330] [moveit_7

## 🧠 다중 파이프라인 계획 (multi-pipeline)

MoveIt2 는 한 목표에 대해 **여러 플래너를 동시에 돌리고 그중 되는(또는 제일 좋은) 걸 채택**할 수 있어.
여기선 런치/params 에 정의된 두 파이프라인을 같이 던져:

- `ompl_rrtc` — OMPL RRTConnect (샘플링 기반, 뭐든 잘 푸는 범용 플래너)
- `pilz_ptp` — Pilz PTP (관절공간 직선 보간, 산업용 예측 가능한 궤적)

둘을 동시에 계획하면, 하나가 막혀도 다른 하나가 답을 낼 확률이 올라가. 아래 셀은 리포 ex03 과 동일하게
**다중 파이프라인 params 가 없으면 단일 계획으로 폴백**하도록 `try/except` 로 감쌌어 (환경에 따라
params 미로딩이면 조용히 일반 `plan()` 으로 떨어짐).

In [19]:
arm.set_start_state_to_current_state()
arm.set_goal_state(configuration_name="home")
try:
    multi = MultiPipelinePlanRequestParameters(robot, ["ompl_rrtc", "pilz_ptp"])
    result = arm.plan(multi_plan_parameters=multi)
except Exception as exc:  # 파이프라인 params 미설정 시 단일 계획으로 폴백
    logger.warn(f"다중 파이프라인 불가({exc}) — 단일 계획 폴백")
    arm.set_start_state_to_current_state()
    arm.set_goal_state(configuration_name="home")
    result = arm.plan()
if result:
    robot.execute(result.trajectory, controllers=[])
    print("실행 완료")
else:
    logger.error("계획 실패")

[WARN] [1784809485.118487384] [moveit_py]: Parameter 'ompl_rrtc.plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[WARN] [1784809485.118516941] [moveit_py]: Parameter 'pilz_ptp.plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809485.118710857] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809485.118753019] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809485.118771606] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809485.118781298] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809485.118783267] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809485.118796653] [moveit_786748536.mov

실행 완료


[INFO] [1784809487.329651703] [moveit.simple_controller_manager.follow_joint_trajectory_controller_handle]: Controller 'arm_controller' successfully finished
[INFO] [1784809487.343794650] [moveit_786748536.moveit.ros.trajectory_execution_manager]: Completed trajectory execution with status SUCCEEDED ...


## 🎬 나만의 시퀀스 만들기

한 줄씩 찔러보는 게 익숙해졌으면, 이제 **리스트로 엮어서** 동작을 스크립트처럼 돌려볼 수 있어.
아래는 목표들의 리스트를 순회하면서 순서대로 실행하는 아주 작은 예시야. `steps` 리스트만 고치면
너만의 안무를 커널 안에서 즉석으로 만들 수 있어 — 역시 재빌드/재시작 없이.

In [20]:
# ("종류", 값) 튜플 리스트를 순서대로 실행. 값만 바꿔서 나만의 안무를 짜봐.
steps = [
    ("named", "home"),
    ("grip", "gripper_open"),
    ("joints", {"joint1": 0.4, "joint2": 0.3}),
    ("grip", "gripper_close"),
    ("joints", {"joint1": -0.4, "joint2": 0.3}),
    ("named", "home"),
]

for i, (kind, val) in enumerate(steps, 1):
    print(f"[{i}/{len(steps)}] {kind}: {val}")
    if kind == "named":
        ok = go_named(val)
    elif kind == "joints":
        ok = go_joints(**val)
    elif kind == "grip":
        ok = grip(val)
    else:
        print(f"  알 수 없는 종류: {kind}")
        continue
    if not ok:
        print("  → 실패, 시퀀스 중단")
        break
print("시퀀스 끝")

[1/6] named: home


[WARN] [1784809487.347755099] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809487.347862869] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809487.347893503] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809487.347896828] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809487.347904824] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809487.347913953] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809487.347958118] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809487.348118004] [moveit_7

[2/6] grip: gripper_open


[WARN] [1784809487.929148953] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809487.929306015] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809487.929368861] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809487.929377287] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809487.929393959] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809487.929412001] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809487.929470398] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'gripper' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809487.929668360] [move

[3/6] joints: {'joint1': 0.4, 'joint2': 0.3}


...

[5/6] joints: {'joint1': -0.4, 'joint2': 0.3}


[WARN] [1784809493.739382582] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809493.739585336] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809493.739620584] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809493.739628636] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809493.739644596] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809493.739663112] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809493.739723920] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809493.739915942] [moveit_7

[6/6] named: home


[WARN] [1784809496.074252751] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784809496.074408522] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784809496.074450557] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784809496.074459933] [moveit_786748536.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784809496.074479667] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784809496.074500438] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784809496.074564517] [moveit_786748536.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will use defaults instead.
[INFO] [1784809496.074761100] [moveit_7

시퀀스 끝


## 🏁 정리 — 개발 루프의 승리

- **셀 고침 → `Shift+Enter` → 즉시.** 커널이 살아있는 한 `move_group`/컨트롤러는 안 죽고,
  파이썬 로직은 셀에서 즉석 재정의돼. `colcon` 재빌드도 `ros2 launch` 재시작도 없어. 이게 이 노트북의 전부야.
- 🔒 **설정 셀(맨 위 두 셀)은 다시 실행하지 마.** 헬퍼/놀이터 셀은 얼마든지 재실행해도 되지만,
  `MoveItPy` 생성 셀은 커널당 한 번뿐. 새로 시작하려면 커널 재시작.
- 💀 커널을 끌 때 뜨는 **"Kernel died"** 는 MoveItPy 종료자(destructor)의 **알려진 양성 이슈**야.
  작업은 이미 다 끝난 뒤라 무시해도 돼.

### 실물 로봇으로 넘어갈 때 ⚠️

여기 있는 함수(`go_named`/`go_joints`/`grip`)는 **실물 Piper 에서도 그대로 동작**해 —
mock 대신 실물 스택을 런치하면 끝이야. 단, 실물은 **속도 스케일링을 낮게** 두고 팔에서 눈을 떼지 말 것,
그리고 첫 동작 전엔 반드시 **리포의 실물 로봇 체크리스트**를 따를 것. 이 노트북은 mock 기준 놀이터라
안전 절차는 거기서 다룬다.